In [7]:
from google.colab import drive
drive.mount('/content/drive')

import h5py, re, json, time
DATA_DIR = "/content/drive/MyDrive/cs639FM"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install anthropic -q

import anthropic
client = anthropic.Anthropic(api_key="")

In [8]:
# Load just problems 8-10 for testing
f = h5py.File(f"{DATA_DIR}/shard_avni.h5", "r")

problems = []
for key in ["problem_8", "problem_9", "problem_10"]:
    p = f[key]
    cot = p["cot_trace"][()].decode()
    paras = [x.strip() for x in re.split(r'\n\s*\n', cot) if x.strip()]
    problems.append({
        "problem_id": key,
        "paragraphs": paras,
        "num_paragraphs": len(paras),
        "num_hidden_states": int(p["hidden_states"].shape[0]),
        "correct": int(p["label"][()]),
        "truncated": int(p["truncated"][()])
    })
    print(f"{key}: {len(paras)} paras, correct={p['label'][()]}, truncated={p['truncated'][()]}")

f.close()

problem_8: 47 paras, correct=1, truncated=0
problem_9: 721 paras, correct=0, truncated=1
problem_10: 348 paras, correct=0, truncated=1


In [14]:
##load full problems
f = h5py.File(f"{DATA_DIR}/shard_avni.h5", "r")

problems = []
for key in sorted(f.keys(), key=lambda x: int(x.split('_')[1])):
    p = f[key]
    cot = p["cot_trace"][()].decode()
    paras = [x.strip() for x in re.split(r'\n\s*\n', cot) if x.strip()]
    problems.append({
        "problem_id": key,
        "paragraphs": paras,
        "num_paragraphs": len(paras),
        "num_hidden_states": int(p["hidden_states"].shape[0]),
    })

f.close()
print(f"Loaded {len(problems)} problems")

Loaded 120 problems


In [9]:
def label_paragraphs_api(paragraphs, chunk_size=40):
    """Label paragraphs using Claude API in chunks."""
    all_labels = []

    for start in range(0, len(paragraphs), chunk_size):
        chunk = paragraphs[start:start + chunk_size]

        span_list = ""
        for i, text in enumerate(chunk):
            span_list += f"\n[SPAN {start + i}]\n{text}\n"

        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=16384,
            temperature=0,
            messages=[{
                "role": "user",
                "content": f"""Classify each numbered span as either STRATEGY or EXECUTION.

- STRATEGY: choosing an approach, planning, reconsidering, deciding what to do, identifying problem type, "wait let me think", "hmm maybe I should", changing direction
- EXECUTION: computing, substituting, simplifying, carrying out algebra, verifying by calculation, stating results

Return ONLY a JSON list like: [{{"span_id": 0, "label": "STRATEGY"}}, ...]

{span_list}

JSON:"""
            }]
        )

        raw = response.content[0].text.strip()
        try:
            start_idx = raw.index('[')
            bracket_count = 0
            end_idx = start_idx
            for i in range(start_idx, len(raw)):
                if raw[i] == '[': bracket_count += 1
                elif raw[i] == ']': bracket_count -= 1
                if bracket_count == 0:
                    end_idx = i + 1
                    break
            parsed = json.loads(raw[start_idx:end_idx])
            all_labels.extend(parsed)
        except:
            for i, text in enumerate(chunk):
                all_labels.append({"span_id": start + i, "label": "UNKNOWN"})

    return all_labels

In [ ]:
save_path = f"{DATA_DIR}/labeled_api.jsonl"

# Resume support
results = []
done_ids = set()
try:
    with open(save_path) as f:
        for line in f:
            r = json.loads(line)
            results.append(r)
            done_ids.add(r["problem_id"])
    print(f"Resuming: {len(results)} already labeled")
except FileNotFoundError:
    pass

for prob in problems:
    if prob["problem_id"] in done_ids:
        continue

    print(f"[{len(results)+1}/{len(problems)}] {prob['problem_id']} "
          f"({prob['num_paragraphs']} paras)...", end=" ")
    start = time.time()

    try:
        labels = label_paragraphs_api(prob["paragraphs"])
        label_map = {l["span_id"]: l["label"] for l in labels}
        paragraph_labels = [label_map.get(i, "UNKNOWN") for i in range(prob["num_paragraphs"])]

        elapsed = time.time() - start
        strat = paragraph_labels.count("STRATEGY")
        exe = paragraph_labels.count("EXECUTION")
        unk = paragraph_labels.count("UNKNOWN")
        print(f"Done ({elapsed:.1f}s, {strat}S/{exe}E/{unk}U)")

        results.append({
            "problem_id": prob["problem_id"],
            "num_paragraphs": prob["num_paragraphs"],
            "num_hidden_states": prob["num_hidden_states"],
            "paragraph_labels": paragraph_labels
        })

        with open(save_path, 'w') as f:
            for r in results:
                f.write(json.dumps(r) + "\n")

    except Exception as e:
        print(f"ERROR: {e}")
        time.sleep(5)

print(f"\nDone! {len(results)} labeled problems saved to {save_path}")

Labeling problem_8 (47 paras)... Done (7.3s, 13S/34E/0U)
Labeling problem_9 (721 paras)... Done (109.6s, 497S/224E/0U)
Labeling problem_10 (348 paras)... Done (57.8s, 167S/181E/0U)

Saved to /content/drive/MyDrive/cs639FM/labeled_api_test.jsonl


In [ ]:
with open(f"{DATA_DIR}/labeled_api.jsonl") as f:
    results = [json.loads(line) for line in f]

total = sum(r["num_paragraphs"] for r in results)
strat = sum(r["paragraph_labels"].count("STRATEGY") for r in results)
exe = sum(r["paragraph_labels"].count("EXECUTION") for r in results)
unk = sum(r["paragraph_labels"].count("UNKNOWN") for r in results)

print(f"Labeled: {len(results)} problems, {total} paragraphs")
print(f"Strategy:  {strat} ({strat/total*100:.1f}%)")
print(f"Execution: {exe} ({exe/total*100:.1f}%)")
print(f"Unknown:   {unk} ({unk/total*100:.1f}%)")

problem_8: 47 paras, 13S/34E/0U (28% strat), correct=1, truncated=0
problem_9: 721 paras, 497S/224E/0U (69% strat), correct=0, truncated=1
problem_10: 348 paras, 167S/181E/0U (48% strat), correct=0, truncated=1
